In [ ]:
import os
import ast
import pandas as pd
import osmnx as ox
import networkx as nx
from joblib import Parallel, delayed
import route_network_analysis as rna



def process_routes(graph_files, row):
    try:
        print(f"Processing {row["city_name"]}",flush=True)
        json_path = os.path.join(
            "experiment_routes", "json_routes", row["city_name"] + str(row["id"]) + ".json"
        )

        filepath = graph_files.loc[graph_files["city_name"] == row["city_name"]][
            "graph_file"
        ].values[0]
        print(filepath)
        graph = ox.load_graphml(filepath)
        graph, _ = rna.street_network_analysis.add_deviation_from_prototypical_weights(
            graph
        )
        graph, _ = rna.street_network_analysis.add_instruction_equivalent_weights(graph)
        graph, _ = rna.street_network_analysis.add_node_degree_weights(graph)
        betweenness_centrality = nx.betweenness_centrality(graph, normalized=True)
        nx.set_node_attributes(graph, betweenness_centrality, "betweenness_centrality")
        ox.save_graphml(graph, filepath)
        # graph = ox.elevation.add_node_elevations_google(graph, api_key=google_key,pause=0.1)
        # graph.graph['node_attributes'] = ast.literal_eval(graph.graph['node_attributes']).append('elevation')

        old_complexity = row["sum_decision_complexity"]
        route_nodes = ast.literal_eval(row["nodes"])

        wstring = "length"
        if row["weight"] == "least_decision_complex":
            route_nodes = route_nodes.reverse()
            wstring = "decision_complexity"

        new_route_od_pair = rna.od_pair.from_route(graph, route_nodes, wstring)
        new_route_df = new_route_od_pair.get_odpair_df()
        new_route_df["old_complexity"] = old_complexity
        new_route_df["id"] = row["id"]
        new_route_df["complexity_difference"] = (
            old_complexity - new_route_od_pair.path.complexity
        )
        new_route_df["route_exp_condition"] = row["condition"]

        new_route_df.to_json(json_path,indent=4,default_handler=str)
        print("finished with route id", row["id"])
        return new_route_df
    except Exception as e:
        print(f"failed processing {row["city_name"]}, error: {e}") 
    

route_data = pd.read_csv(os.path.join("experiment_routes", "route_data.csv"))
graph_files = pd.read_csv(os.path.join("experiment_routes", "graph_city_dicts.csv"))
graph_files["graph_file"] = graph_files["graph_file"].str.replace("\\", "/")

# comparison_dicts = Parallel(n_jobs=4, backend="loky")(
#    delayed(compare_routes)(graph_files, row) for _, row in route_data.iterrows()
# )

odpair_dfs = Parallel(n_jobs=8, backend="loky")(
    delayed(process_routes)(graph_files, row) for _, row in route_data.iterrows()
)


df = pd.concat(odpair_dfs)
df.to_csv(os.path.join("experiment_routes/experiment_route_data.csv"))
